# Aula 03 - Correlação, Covariância e o Argumento Matemático da Diversificação

Módulo 08 AdmTech, Computação · Semana 5 · Sprint 3 · Prof. José Romualdo

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josercf/2026-2B-M08/blob/main/notebooks/aula03.ipynb)

## O que este notebook é

Laboratório executável da Aula 03, que alimenta o artefato A3 (Motor de Simulação de Monte Carlo) com a matriz de covariância anualizada da cesta. Roda no Colab pelo link acima ou localmente a partir do repositório clonado. Sem rede, cada célula de carga usa o snapshot versionado em `dados/`. Nenhuma carteira da Ponderada 1 aparece aqui: os exemplos usam a cesta de referência do acervo.

## 1. Os retornos e a janela comum

O tratamento de ausência é o do módulo inteiro, registrado na ADR-009 (`docs/adrs/ADR-009-tratamento-de-ausencia-unico-com-janela-comum.md`):

- retorno simples diário do `Adj Close`, com a lacuna isolada preenchida pelo último preço (`ffill`) e nada preenchido antes da primeira cotação; a GOLD11 só tem retorno a partir de 14/12/2020;
- a medida de **um ativo** usa a série inteira dele, como na tabela de volatilidade da Aula 02;
- a análise **conjunta** (uma matriz, um par, uma carteira) usa a **janela comum** dos ativos envolvidos, os pregões em que todos têm retorno.

O notebook roda no Colab e não importa `tools/retornos.py` do repositório; a função `janela_comum` abaixo repete a mesma regra em uma linha.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DADOS_LOCAL = os.path.join("..", "dados")
DADOS_BRUTOS = "https://raw.githubusercontent.com/josercf/2026-2B-M08/main/dados/"
PREGOES_ANO = 252


def ler_csv(nome, **kwargs):
    """Lê um CSV de dados/: do disco quando o repositório está clonado, da URL bruta no Colab."""
    local = os.path.join(DADOS_LOCAL, nome)
    if os.path.exists(local):
        return pd.read_csv(local, **kwargs)
    if os.environ.get("M08_OFFLINE") == "1":
        raise FileNotFoundError("M08_OFFLINE=1 e %s não existe localmente" % local)
    return pd.read_csv(DADOS_BRUTOS + nome, **kwargs)


precos = ler_csv("precos_ajustados.csv", index_col="data", parse_dates=["data"]).ffill()
retornos = precos.pct_change(fill_method=None).iloc[1:]  # um ativo: série inteira dele


def janela_comum(rets, tickers=None):
    """ADR-009: análise conjunta usa os pregões em que todos os ativos pedidos têm retorno."""
    return (rets if tickers is None else rets[list(tickers)]).dropna()


comum = janela_comum(retornos)  # a matriz da cesta parte daqui
print("pregões com retorno:", len(retornos))
print("retornos por ativo (GOLD11 começa depois):")
print(retornos.count().tail(3))
print("janela comum dos 15 ativos: %d retornos, de %s a %s"
      % (len(comum), comum.index[0].date(), comum.index[-1].date()))

## 2. O risco de dois ativos

A variância de uma carteira com peso `w` no ativo A e `1 - w` no ativo B é

`w² σa² + (1 − w)² σb² + 2 w (1 − w) ρ σa σb`

Só o último termo depende da correlação `ρ`. Com `ρ = 1`, o risco da carteira é a média ponderada das duas volatilidades; com qualquer `ρ` menor, fica abaixo dela. A célula seguinte refaz a conta do Peer Instruction de abertura.

In [ ]:
def risco_dois_ativos(peso, vol_a, vol_b, rho):
    variancia = (peso ** 2 * vol_a ** 2 + (1 - peso) ** 2 * vol_b ** 2
                 + 2 * peso * (1 - peso) * rho * vol_a * vol_b)
    return np.sqrt(max(variancia, 0.0))


for rho in [1.0, 0.5, 0.0, -1.0]:
    print("rho = %+.1f  ->  risco da carteira meio a meio: %.2f%%" % (rho, 100 * risco_dois_ativos(0.5, 0.30, 0.30, rho)))

In [ ]:
a, b = "PETR4", "IVVB11"
par = janela_comum(retornos, [a, b])  # um par: janela comum dos dois ativos
vol = par.std() * np.sqrt(PREGOES_ANO)
ret = par.mean() * PREGOES_ANO
pesos = np.linspace(0, 1, 101)

fig, eixo = plt.subplots(figsize=(10, 4))
for rho in [1.0, 0.5, 0.0, -0.5, -1.0]:
    eixo.plot(100 * pesos, [100 * risco_dois_ativos(w, vol[a], vol[b], rho) for w in pesos], label="ρ = %+.1f" % rho)
eixo.set_xlabel("peso em %s (%%)" % a)
eixo.set_ylabel("risco anual (%)")
eixo.set_title("%s com %s: o risco da carteira conforme a correlação" % (a, b))
eixo.legend()
plt.show()

## 3. A matriz de correlação da cesta

A correlação é a covariância dividida pelo produto dos dois desvios padrão. Ela não tem unidade, fica sempre entre −1 e 1 e não muda com o período, e por isso é a medida que circula num relatório ao comitê. Por padrão, o `DataFrame.corr` usa em cada par os pregões em que os dois ativos têm retorno, e cada par fica numa janela diferente. A matriz desta aula parte da janela comum dos 15 ativos (ADR-009), e todos os pares usam os mesmos 1.423 retornos.

In [ ]:
correlacao = comum.corr()

fig, eixo = plt.subplots(figsize=(8, 7))
imagem = eixo.imshow(correlacao, cmap="PRGn_r", vmin=-1, vmax=1)
eixo.set_xticks(range(len(correlacao)), correlacao.columns, rotation=90)
eixo.set_yticks(range(len(correlacao)), correlacao.columns)
fig.colorbar(imagem, ax=eixo, shrink=0.8)
eixo.set_title("Correlação dos retornos diários da cesta, janela comum dos 15 ativos")
plt.show()

In [ ]:
acima = np.triu(np.ones(correlacao.shape, dtype=bool), 1)
pares = correlacao.where(acima).stack().dropna().sort_values()
print("pares:", len(pares), " média: %.2f" % pares.mean())
print("\nos três mais correlacionados:")
print(pares.tail(3).round(2))
print("\nos três menos correlacionados:")
print(pares.head(3).round(2))

### Pergunta para a dupla

1. O par mais correlacionado é formado por dois bancos privados. Que risco em comum explica isso?
2. Os pares negativos têm todos um de dois ativos. Quais, e por que eles tendem a se mover contra a bolsa brasileira?

<details>
<summary>Resposta</summary>

1. ITUB4 e BBDC4 respondem aos mesmos fatores: juros, inadimplência e regulação bancária. Quando um desses fatores muda, os dois se movem juntos.
2. GOLD11, o ETF de ouro, e IVVB11, a bolsa americana em reais. O ouro costuma ser procurado quando a bolsa cai, e o IVVB11 sobe em reais quando o dólar sobe, o que costuma acontecer quando a bolsa brasileira cai.

</details>

## 4. A matriz de covariância anualizada

A covariância diária vezes 252 dá a covariância anual, da mesma forma que a variância diária vezes 252 dá a variância anual. A diagonal da matriz anual é a variância de cada ativo na janela comum, e a raiz dela é a volatilidade anual nessa janela. É esta matriz que o motor de Monte Carlo do A3 e o otimizador da Aula 04 consomem.

Um erro comum é anualizar a correlação junto: ela não tem período, e multiplicada por 252 passa de 1, o que não tem significado.

In [ ]:
cov_anual = comum.cov() * PREGOES_ANO
excerto = ["PETR4", "VALE3", "ITUB4", "IVVB11"]
print(cov_anual.loc[excerto, excerto].round(4))
print("\nraiz da diagonal (volatilidade anual na janela comum):")
print(np.sqrt(np.diag(cov_anual.loc[excerto, excerto])).round(4))

In [ ]:
# Duas verificações antes de entregar a matriz ao motor do A3.
assert np.allclose(cov_anual, cov_anual.T), "matriz não simétrica"
autovalores = np.linalg.eigvalsh(cov_anual.to_numpy())
assert autovalores.min() > 0, "matriz não positiva definida"
print("simétrica e positiva definida; menor autovalor: %.6f" % autovalores.min())

# Exportação para o motor: grava fora do repositório, numa pasta temporária.
import tempfile
destino = os.path.join(tempfile.gettempdir(), "covariancia_anual.csv")
cov_anual.to_csv(destino)
print("gravado em", destino)

### O custo da janela comum

A raiz da diagonal da PETR4 não repete a volatilidade da Aula 02. A janela comum dos 15 ativos só começa quando a GOLD11 passa a ter retorno, e os pregões anteriores, março de 2020 entre eles, ficam fora da matriz. É o efeito de descartar datas que a Aula 01 mostrou. A célula seguinte mede esse custo e a alternativa de tirar a GOLD11 da cesta; o ADR do estimador de covariância do A3 registra qual das duas o grupo escolhe.

In [ ]:
sem_ouro = janela_comum(retornos, [t for t in retornos.columns if t != "GOLD11"])
for nome, rets in [("série inteira da PETR4", retornos[["PETR4"]].dropna()),
                   ("janela comum dos 15", comum),
                   ("janela comum dos 14, sem GOLD11", sem_ouro)]:
    print("%-32s %4d retornos desde %s, %2d pregões em março de 2020, PETR4 com %.2f%% ao ano"
          % (nome, len(rets), rets.index[0].date(), len(rets.loc["2020-03-01":"2020-03-31"]),
             100 * rets["PETR4"].std() * np.sqrt(PREGOES_ANO)))

## 5. Correlação móvel e o que acontece em crise

A correlação da janela inteira é uma média de períodos calmos e de crises. Calculada em janelas de 63 pregões, cerca de um trimestre, ela mostra quando os pares passam a andar juntos.

In [ ]:
fig, eixo = plt.subplots(figsize=(11, 4))
for a, b in [("ITUB4", "BBDC4"), ("PETR4", "IVVB11")]:
    par = janela_comum(retornos, [a, b])
    par[a].rolling(63).corr(par[b]).plot(ax=eixo, label="%s com %s" % (a, b))
eixo.axvspan("2020-03-01", "2020-03-31", alpha=0.2)
eixo.axhline(0, linewidth=0.8)
eixo.set_ylabel("correlação em 63 pregões")
eixo.legend()
plt.show()

In [ ]:
def correlacao_media(rets):
    c = rets.corr()
    return c.where(np.triu(np.ones(c.shape, dtype=bool), 1)).stack().dropna().mean()


# Janela comum dos 14 ativos sem a GOLD11: com ela, a janela começaria em 14/12/2020 e perderia março de 2020.
sem_gold = janela_comum(retornos, [t for t in retornos.columns if t != "GOLD11"])
print("correlação média dos pares, janela inteira: %.2f" % correlacao_media(sem_gold))
print("correlação média dos pares, março de 2020:  %.2f (%d pregões)"
      % (correlacao_media(sem_gold.loc["2020-03"]), len(sem_gold.loc["2020-03"])))

## 6. Encolhimento de Ledoit-Wolf

Uma matriz de covariância de 15 ativos estima 120 números. Com poucos pregões, parte deles é ruído, e um otimizador aposta nesse ruído. O estimador de Ledoit-Wolf (2004) mistura a matriz amostral com um alvo simples, a identidade multiplicada pela variância média, e calcula dos próprios dados a intensidade da mistura. A função abaixo reproduz o `sklearn.covariance.LedoitWolf` em NumPy, para o notebook rodar sem scikit-learn.

In [ ]:
def ledoit_wolf(matriz):
    """Devolve (covariância encolhida, intensidade do encolhimento entre 0 e 1)."""
    x = np.asarray(matriz, dtype=float)
    n, p = x.shape
    x = x - x.mean(axis=0)
    amostral = x.T @ x / n
    alvo = np.trace(amostral) / p * np.eye(p)
    d2 = np.sum((amostral - alvo) ** 2)
    normas = np.sum(x ** 2, axis=1)
    quadratica = np.einsum("ij,jk,ik->i", x, amostral, x)
    b2_barra = np.sum(normas ** 2 - 2 * quadratica + np.sum(amostral ** 2)) / n ** 2
    intensidade = min(b2_barra, d2) / d2
    return intensidade * alvo + (1 - intensidade) * amostral, intensidade


completos = comum  # janela comum dos 15 ativos, a mesma da matriz
for n in [63, 126, 252, len(completos)]:
    trecho = completos.iloc[-n:]
    _, intensidade = ledoit_wolf(trecho.to_numpy())
    print("últimos %4d pregões, desde %s: encolhimento de %.2f%%" % (n, trecho.index[0].date(), 100 * intensidade))

## 7. Uma carteira só de bancos

Três ativos em partes iguais, medidos na janela comum dos três (ADR-009). Compare a volatilidade da carteira com a média das três volatilidades: a diferença é o que a diversificação eliminou.

In [ ]:
def carteira_igual(tickers):
    cov = (janela_comum(retornos, tickers).cov() * PREGOES_ANO).to_numpy()
    w = np.full(len(tickers), 1 / len(tickers))
    vol_carteira = np.sqrt(w @ cov @ w)
    media_vols = w @ np.sqrt(np.diag(cov))
    return vol_carteira, media_vols, 1 - vol_carteira / media_vols


for tickers in (["ITUB4", "BBDC4", "BBAS3"], ["ITUB4", "PETR4", "IVVB11"]):
    v, mv, red = carteira_igual(tickers)
    print("%-24s carteira %.2f%%, média %.2f%%, redução %.2f%%" % (", ".join(tickers), 100 * v, 100 * mv, 100 * red))

### Os oito pares da escolha coletiva, meio a meio

No Bloco 6 a sala classificou oito pares por uma heurística exploratória (ρ a partir de 0,60 concentra, de 0,30 a 0,60 é intermediário, abaixo de 0,30 diversifica). Os limiares não decidem nada sozinhos: o quanto a carteira reduz o risco depende também dos pesos e das volatilidades, e a correlação muda com a janela. A célula seguinte mede cada par com 50% em cada ativo, na janela comum dos dois (ADR-009), e mostra o início da janela, o número de retornos, a correlação nessa janela e a redução contra a média ponderada das volatilidades.

In [ ]:
pares_escolha = [("ITUB4", "BBDC4"), ("BBAS3", "BOVA11"), ("ITUB4", "BBAS3"), ("PETR4", "BOVA11"),
                 ("PETR4", "VALE3"), ("ITUB4", "IVVB11"), ("PETR4", "IVVB11"), ("BOVA11", "GOLD11")]
for a, b in pares_escolha:
    par = janela_comum(retornos, [a, b])
    v, mv, red = carteira_igual([a, b])
    print("%-15s %4d retornos desde %s  rho %+.2f  carteira %.2f%%  média %.2f%%  redução %.2f p.p. (%.2f%%)"
          % (a + " e " + b, len(par), par.index[0].date(), par[a].corr(par[b]),
             100 * v, 100 * mv, 100 * (mv - v), 100 * red))

## 8. Desafio para o A3

a. Escolher duas janelas distintas do período (por exemplo, antes e depois de 2023) e completar `matriz_na_janela`, usando a janela comum dos ativos dentro de cada período (ADR-009). Com a GOLD11, a primeira janela do exemplo começa em 14/12/2020; diga no ADR se isso pesa.
b. Medir quanto cada par se move entre as duas janelas e listar os cinco pares que mais mudaram.
c. Registrar no ADR do estimador de covariância qual janela o grupo vai usar no motor e o número que justifica a escolha.

In [ ]:
def matriz_na_janela(rets, inicio, fim):
    """Desafio: devolva a matriz de correlação dos retornos entre inicio e fim, na janela comum."""
    return None


janela_1 = matriz_na_janela(retornos, "2019-01-01", "2022-12-31")
janela_2 = matriz_na_janela(retornos, "2023-01-01", "2026-08-31")
if janela_1 is None or janela_2 is None:
    print("Desafio em aberto: complete matriz_na_janela e rode de novo.")
else:
    diferenca = (janela_2 - janela_1).where(np.triu(np.ones(janela_1.shape, dtype=bool), 1)).stack().dropna()
    print(diferenca.abs().sort_values(ascending=False).head(5).round(2))